# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide to loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema, accessible [here](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available **record sets**, their **fields** and **columns**, and their `@id`s.

In [ ]:
# Display all record sets, their IDs, and fields
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in this dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name} (@id: {rs.id})")
        print("  Fields:")
        for field in rs.fields:
            print(f"   - {field.name} (@id: {field.id})")
        print("  Columns:")
        for column in rs.columns:
            print(f"   - {column.name} (@id: {column.id})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract all available record sets (referenced by their `@id`), then explore the columns for the first record set as an example.

In [ ]:
# Gather record set IDs
record_set_objs = list(dataset.record_sets)
record_set_ids = [rs.id for rs in record_set_objs]

dataframes = {}
for record_set_id in record_set_ids:
    # Records generator for each record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns in record set '@id': {first_record_set_id}")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()
else:
    print('No record sets found to extract records.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

The EDA will focus on a numeric field (by `@id`), such as 'log_likelihood', if present, filtering, normalizing and grouping as examples.

In [ ]:
# Select record set and numeric field by @id for EDA
if record_set_ids and dataframes[record_set_ids[0]].shape[1] > 0:
    # Try to pick a likely record set and numeric field
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Print a sample of columns for manual inspection
    print(f"Available columns in record set {record_set_id}:\n", df.columns.tolist())
    
    # Try to select a likely numeric field (commonly found: log likelihood, coefficient, p-value, etc.)
    import re
    numeric_candidates = [col for col in df.columns if re.search(r'log_likelihood|coefficient|value|score|num|count|error|std|pval', col, re.I)]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = 10
        # Remove NA before filtering
        filtered_df = df.loc[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        numeric_vals = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
        filtered_df[f"{numeric_field}_normalized"] = (numeric_vals - numeric_vals.mean()) / numeric_vals.std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical/group key
        group_candidates = [col for col in df.columns if col.lower() not in [numeric_field.lower()]]
        if group_candidates:
            group_field = group_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"\nGrouped data by {group_field}:")
                print(grouped_df.head())
            else:
                print(f"Group field {group_field} not found in filtered DataFrame.")
        else:
            print("No suitable group field found for grouping.")
    else:
        print('No numeric-like columns found for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use previous df, numeric_field and group_field if they exist
try:
    if numeric_candidates and not filtered_df.empty:
        plt.figure(figsize=(8, 5))
        sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field} (> {threshold}) in '{record_set_id}' record set")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        if 'group_field' in locals() and group_field in filtered_df.columns:
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()
except Exception as e:
    print(f"Visualization skipped due to error: {e}")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using `mlcroissant`:
- Loaded schema metadata and displayed dataset info.
- Examined available record sets, fields, and columns by their `@id`s.
- Extracted records from one or more record sets into DataFrames.
- Performed exploratory data analysis including filtering, normalization, and grouping by attributes using column `@id`s.
- Visualized distributions in the dataset.

Further analysis may involve deep data cleaning (depending on domain knowledge for record set contents), field selection, and more domain-specific questions around predictors of knowledge adoption. Be sure to always reference schema entities by their `@id` for clarity and reproducibility.